**Cell 2 — Algorithmic Overview**

- Mixture model: we fit a finite mixture model to mixed-type phenotype data (continuous, binary, categorical). The model assumes observations come from a mixture of K latent components (classes).
- Estimation (EM-like): StepMix implements an EM-style algorithm to estimate component parameters and posterior responsibilities. Initialization matters; `n_init` controls random restarts.
- Measurement descriptor: mixed-data types are described by a `measurement` descriptor (which tells the model which columns are continuous, binary, or categorical). This allows correct likelihood terms for each feature type.
- Structural covariate model: we include covariates (`sex`, `age_at_eval_years`) in the model as predictors of component membership. This separates measurement (features defining components) from structural effects (covariate influence).
- Feature enrichment: after assigning samples to classes, we test each feature for enrichment (binary features → binomial test; continuous/categorical → t-test). We correct p-values for multiple testing (FDR BH) and compute effect sizes (fold enrichment or Cohen's d).

In [ ]:
# Cell 3 — Imports and environment setup
import os
import sys
from pathlib import Path

# Ensure imports find local module `utils.py`
repo_root = Path.cwd()
phenotype_module_path = repo_root / 'PhenotypeClasses'
if str(phenotype_module_path) not in sys.path:
    sys.path.insert(0, str(phenotype_module_path))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

# StepMix (mixture model) — the same library used in GFMM.py
from stepmix.stepmix import StepMix
from stepmix.utils import get_mixed_descriptor

# Local helpers (from the repo)
from utils import split_columns, get_feature_enrichments

# Notebook display settings
%matplotlib inline
sns.set(style='whitegrid')
plt.rcParams.update({'figure.dpi': 150})

print('Loaded libraries — StepMix available:', 'StepMix' in globals())

**Cell 4 — Data loading**

We read the phenotype matrix (`spark_5392_cohort.txt`) used in `GFMM.py`. This file is expected under `PhenotypeClasses/data/` (the same `data/` referenced in the script). The index column is the sample/subject ID.

In [ ]:
# Cell 5 — Load data (adjust path if needed)
data_path = Path('data/spark_5392_cohort.txt')
if not data_path.exists():
    raise FileNotFoundError(f"Data file not found: {data_path.resolve()}")

datadf = pd.read_csv(data_path, sep='\t', index_col=0)
# mirror the script behaviour
datadf = datadf.round()

print('Data shape:', datadf.shape)
print('Columns sample:', list(datadf.columns[:20]))
datadf.head()

**Cell 6 — Data preparation and descriptors**

We separate covariates (structural variables) from measurement variables. `split_columns()` inspects saved lists of continuous/binary/categorical features and returns three lists. `get_mixed_descriptor()` converts the pandas DataFrame into a numeric array plus a `measurement` descriptor the StepMix model understands.

Notes on descriptor:
- Continuous features: modeled with Gaussian likelihood
- Binary features: modeled with Bernoulli (binomial) likelihood
- Categorical features: modeled as multinomial / categorical likelihood

In [ ]:
# Cell 7 — Prepare X (measurements) and Z (covariates)
Z_p = datadf[['sex', 'age_at_eval_years']].copy()
X = datadf.drop(['sex', 'age_at_eval_years'], axis=1)

# Ask utils to split features by type — this uses pickled lists in data/
continuous_columns, binary_columns, categorical_columns = split_columns(list(X.columns))
print('Counts — continuous:', len(continuous_columns), 'binary:', len(binary_columns), 'categorical:', len(categorical_columns))

# Convert to mixed representation used by StepMix
mixed_data, mixed_descriptor = get_mixed_descriptor(
    dataframe=X,
    continuous=continuous_columns,
    binary=binary_columns,
    categorical=categorical_columns
)

print('Mixed data shape:', mixed_data.shape)
print('Measurement descriptor (first 10 entries):', mixed_descriptor[:10])

**Cell 8 — Model specification and fitting**

Key StepMix arguments used:
- `n_components`: number of mixture components/clusters
- `measurement`: the measurement descriptor created above
- `structural='covariate'`: include covariates to model class membership
- `n_steps`: number of EM optimization steps inside semi-supervised workflows (left as in script)
- `n_init`: number of random restarts — increases chance of finding good optima

Best practices:
- Run several values of `n_components` and compare model selection criteria or cluster interpretability.
- Use multiple `n_init` seeds and keep the best run by log-likelihood.
- If data is large, reduce `n_init` or use a subset for hyperparameter search.

In [ ]:
# Cell 9 — Fit the mixture model (adjust params as needed)
ncomp = 4  # number of clusters
n_init = 200  # number of random restarts from GFMM.py — may be heavy

model = StepMix(
    n_components=ncomp,
    measurement=mixed_descriptor,
    structural='covariate',
    n_steps=1,
    n_init=n_init
)

print('Fitting StepMix model (this may take a while depending on n_init)...')
model.fit(mixed_data, Z_p)
print('Model fitted. Converged:', getattr(model, 'converged_', 'unknown'))

# Predict cluster labels (hard assignments)
mixed_data['mixed_pred'] = model.predict(mixed_data)
mixed_data['age'] = datadf['age_at_eval_years']

# Save labeled data for downstream use
out_path = Path('data/SPARK_5392_ninit_cohort_GFMM_labeled.csv')
mixed_data.to_csv(out_path)
print('Saved labeled data to', out_path)

# Quick class distribution
print('\nClass counts:')
print(mixed_data['mixed_pred'].value_counts().sort_index())

**Cell 10 — Feature enrichment: tests and effect sizes (detailed)**

After assigning samples to classes, `get_feature_enrichments()` performs per-feature hypothesis testing to identify which features are enriched (higher in a class) or depleted (lower in a class):

- Binary features:
  - Uses a binomial test comparing the number of "successes" in a class to the global probability.
  - Fold enrichment = (proportion in class) / (global proportion).
  - Two-directional tests: `alternative='greater'` identifies enriched features, `alternative='less'` identifies depleted features.

- Continuous/categorical features:
  - Uses two-sample Welch's t-test (`ttest_ind` with `equal_var=False`) comparing the target class to the pooled other classes.
  - Computes Cohen's d as an effect size (difference in means divided by pooled SD).

- Multiple testing:
  - P-values from tests are corrected by the Benjamini-Hochberg procedure (`fdr_bh`) using `statsmodels.stats.multitest.multipletests`.
  - Significant features are those with corrected p-value < 0.05.

- Outputs:
  - `pval_classification_df`: matrix of -1/0/1 indicating depleted/no-effect/enriched per class
  - `df_enriched_depleted`: summary table with enriched/depleted p-values per class
  - `fold_enrichments`: effect size table (fold enrichment for binary, Cohen's d for continuous)

In [ ]:
# Cell 11 — Compute feature enrichments and inspect top features
results = get_feature_enrichments(mixed_data, summarize=True)
# The function returns 6 items when summarize=True
pval_class_df, feature_sig_high, feature_sig_low, feature_vector, df_enriched_depleted, fold_enrichments = results

print('Enriched/depleted dataframe shape:', df_enriched_depleted.shape)
df_enriched_depleted.head(10)

**Cell 12 — Visualizations: class proportions and demographics**

The original script creates:
- Pie chart of class proportions (publication-quality formatting)
- Age density plots per class
- Sex proportion bars per class

We'll reproduce these here and display them inline.

In [ ]:
# Cell 13 — Pie chart of class proportions
fig, ax = plt.subplots(figsize=(5,5))
colors = ['#FBB040','#39B54A','#27AAE1','#EE2A7B']
ax = mixed_data['mixed_pred'].value_counts().sort_index().plot.pie(
    autopct='%1.0f%%', startangle=90, colors=colors, labels=None
)
for patch in ax.patches:
    patch.set_alpha(0.85)
plt.title('Class proportions')
plt.ylabel('')
plt.show()

# Cell 14 — Age distributions per class
from matplotlib import gridspec

fig, axes = plt.subplots(2, 2, figsize=(10, 6))
for cls, ax in enumerate(axes.flatten()):
    sns.kdeplot(mixed_data.loc[mixed_data['mixed_pred']==cls, 'age'].dropna(),
                shade=True, ax=ax, linewidth=2, color=colors[cls])
    ax.set_title(f'Class {cls}')
    ax.set_xlabel('Age')
    ax.set_ylabel('Density')
plt.suptitle('Age distributions by predicted class')
plt.tight_layout()
plt.show()

# Cell 15 — Sex distribution by class
mixed_demo = mixed_data.copy()
# If sex is 0/1, map to strings for plotting
if set(mixed_demo['sex'].unique()) <= {0,1}:
    mixed_demo['sex_str'] = mixed_demo['sex'].replace({0: 'Female', 1: 'Male'})
else:
    mixed_demo['sex_str'] = mixed_demo['sex'].astype(str)

grouped = mixed_demo.groupby('mixed_pred')['sex_str'].value_counts(normalize=True).reset_index(name='proportion')
plt.figure(figsize=(6,4))
sns.barplot(data=grouped, x='mixed_pred', y='proportion', hue='sex_str')
plt.title('Sex distribution by class')
plt.xlabel('Predicted class')
plt.ylabel('Proportion')
plt.legend(title='Sex')
plt.show()

**Cell 16 — Summary table generation and visualization explanation**

`generate_summary_table()` in the original script:
- Removes uninformative features (very low fold change or missing across classes)
- Maps each feature to a phenotype category (using an external mapping CSV)
- Computes proportions of enriched/depleted features per category and plots grouped bar charts as well as a horizontal line plot summarizing enrichment across phenotype categories.

This is useful to move from individual feature-level signals to interpretable phenotype category signals.

In [ ]:
# Cell 17 — Generate the summary table and save figures
# The function expects to find pickles and the feature->category mapping under ../PhenotypeValidations/data/
# If files are missing, this cell will raise errors; adapt paths if your repo structure differs.

try:
    generate_summary_table(df_enriched_depleted, fold_enrichments)
    print('Summary figures generated and saved in figures/')
except Exception as e:
    print('generate_summary_table failed:', e)
    print('If this fails, ensure the file ../PhenotypeValidations/data/feature_to_category_mapping.csv exists and data/binary_columns.pkl is present.')

**Cell 18 — Practical notes and recommended next steps**

- Reproducibility: save the model object and random seed. You can call `pickle.dump(model, open('model.pkl','wb'))` after fitting.
- Model selection: run `ncomp` in a small grid (e.g., 2..6) and evaluate by interpretability, cluster size balance, or likelihood-based criteria if available.
- Initialization: reduce `n_init` while experimenting, but increase again for final models.
- Covariate modeling: if you suspect covariates explain most of the structure, try a model without structural covariates to evaluate differences.
- Testing: for small counts in binary tests, binomial test is robust; for continuous features with non-normal distributions, consider non-parametric tests (e.g., Mann-Whitney) and robust effect sizes.

If you'd like, I can:
- Add cells to run hyperparameter sweeps for `ncomp` and collect summary plots
- Save the fitted model object and show reproducible code to reload and predict on new data
- Convert this XML-like notebook into a standard JSON `.ipynb` if you prefer the canonical Jupyter format